In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ========= 1. CONFIG =========
DATA_PATH = Path("mobile_device_usage.csv")  
TARGET_COL = "Action"                      

BATCH_SIZE = 32
LR = 1e-3
EPOCHS = 30
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# ========= 2. LOAD DATA =========
df = pd.read_csv(DATA_PATH)

print("Columns in dataset:")
print(df.columns.tolist())


if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataset columns!")


Columns in dataset:
['Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port', 'Action', 'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent', 'pkts_received']


In [3]:
# ========= 3. SEPARATE FEATURES & TARGET =========
df["Action"] = df["Action"].replace({
    "reset-both": "deny"
})

print(df["Action"].value_counts())

y_raw = df[TARGET_COL].astype(str)

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
if TARGET_COL in numeric_cols:
    numeric_cols.remove(TARGET_COL)

categorical_cols = [c for c in df.select_dtypes(include=["object", "category"]).columns if c != TARGET_COL]

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

X_numeric = df[numeric_cols].copy()
X_categorical = df[categorical_cols].copy()

Action
allow    37640
deny     15041
drop     12851
Name: count, dtype: int64
Numeric columns: ['Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port', 'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent', 'pkts_received']
Categorical columns: []


In [4]:
# ========= 4. ENCODERS & SCALERS =========
# Hedef label encode
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)
print("Target classes:", list(label_encoder.classes_))

# Sayısalları standardize
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric) if len(numeric_cols) > 0 else np.zeros((len(df), 0))

# Kategorikleri label encode + embedding için kardinalite
cat_encoders = {}
categorical_encoded = []
cat_cardinalities = []

for col in categorical_cols:
    le = LabelEncoder()
    vals = le.fit_transform(X_categorical[col].astype(str))
    cat_encoders[col] = le
    categorical_encoded.append(vals)
    cat_cardinalities.append(len(le.classes_))

if len(categorical_encoded) > 0:
    X_cat_encoded = np.vstack(categorical_encoded).T  # [N, num_cat_cols]
else:
    X_cat_encoded = np.zeros((len(df), 0), dtype=np.int64)

X_num = X_numeric_scaled.astype(np.float32)
X_cat = X_cat_encoded.astype(np.int64)
y = y.astype(np.int64)

Target classes: ['allow', 'deny', 'drop']


In [5]:
# ========= . TRAIN / VAL / TEST SPLIT =========
X_num_train, X_num_temp, X_cat_train, X_cat_temp, y_train, y_temp = train_test_split(
    X_num, X_cat, y, test_size=0.3, random_state=42, stratify=y
)

X_num_val, X_num_test, X_cat_val, X_cat_test, y_val, y_test = train_test_split(
    X_num_temp, X_cat_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [6]:
# ========= 6. DATASET & DATALOADER =========
class TabularDataset(Dataset):
    def __init__(self, X_num, X_cat, y):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_num[idx], self.X_cat[idx], self.y[idx]

train_ds = TabularDataset(X_num_train, X_cat_train, y_train)
val_ds   = TabularDataset(X_num_val,   X_cat_val,   y_val)
test_ds  = TabularDataset(X_num_test,  X_cat_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)


In [7]:
# ========= 7. MODEL DEFINITION =========
class TabularModel(nn.Module):
    def __init__(self, num_numeric_features, cat_cardinalities, num_classes, emb_dim_rule=2):
        super().__init__()
        self.num_numeric_features = num_numeric_features
        self.use_cat = len(cat_cardinalities) > 0

        if self.use_cat:
            self.embeddings = nn.ModuleList()
            emb_out_sizes = []
            for card in cat_cardinalities:
                # Küçük kardinaliteler için küçük embedding
                emb_dim = min(emb_dim_rule * int(round(card ** 0.25)), 50)
                emb_dim = max(emb_dim, 2)
                self.embeddings.append(nn.Embedding(num_embeddings=card, embedding_dim=emb_dim))
                emb_out_sizes.append(emb_dim)
            emb_total_dim = sum(emb_out_sizes)
        else:
            self.embeddings = None
            emb_total_dim = 0

        input_dim = num_numeric_features + emb_total_dim

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, x_num, x_cat):
        if self.use_cat and x_cat.numel() > 0:
            embs = []
            for i, emb in enumerate(self.embeddings):
                embs.append(emb(x_cat[:, i]))
            x = torch.cat([x_num] + embs, dim=1)
        else:
            x = x_num
        out = self.net(x)
        return out

model = TabularModel(
    num_numeric_features=X_num.shape[1],
    cat_cardinalities=cat_cardinalities,
    num_classes=num_classes
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


In [8]:
# ========= 8. TRAINING & EVALUATION LOOPS =========
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for X_num_batch, X_cat_batch, y_batch in loader:
        X_num_batch = X_num_batch.to(DEVICE)
        X_cat_batch = X_cat_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(X_num_batch, X_cat_batch)
            loss = criterion(logits, y_batch)
            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * y_batch.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y_batch.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average="macro")

    return avg_loss, acc, f1

best_val_f1 = 0.0
best_state = None

train_losses = []
val_losses = []
train_f1_list = []
val_f1_list = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1 = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc, val_f1 = run_epoch(model, val_loader, optimizer=None)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_f1_list.append(train_f1)
    val_f1_list.append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = model.state_dict()

    print(
        f"Epoch {epoch:03d} | "
        f"Train loss: {train_loss:.4f}, acc: {train_acc:.3f}, f1: {train_f1:.3f} | "
        f"Val loss: {val_loss:.4f}, acc: {val_acc:.3f}, f1: {val_f1:.3f}"
    )

# En iyi modeli yükle
if best_state is not None:
    model.load_state_dict(best_state)

Epoch 001 | Train loss: 0.1188, acc: 0.966, f1: 0.957 | Val loss: 0.0381, acc: 0.991, f1: 0.987
Epoch 002 | Train loss: 0.0529, acc: 0.988, f1: 0.984 | Val loss: 0.0419, acc: 0.991, f1: 0.987
Epoch 003 | Train loss: 0.0472, acc: 0.990, f1: 0.986 | Val loss: 0.1027, acc: 0.991, f1: 0.988
Epoch 004 | Train loss: 0.0453, acc: 0.990, f1: 0.987 | Val loss: 0.1300, acc: 0.992, f1: 0.988
Epoch 005 | Train loss: 0.0442, acc: 0.990, f1: 0.986 | Val loss: 0.0333, acc: 0.992, f1: 0.989
Epoch 006 | Train loss: 0.0412, acc: 0.990, f1: 0.987 | Val loss: 0.0374, acc: 0.992, f1: 0.989
Epoch 007 | Train loss: 0.0445, acc: 0.990, f1: 0.986 | Val loss: 0.2474, acc: 0.992, f1: 0.988
Epoch 008 | Train loss: 0.0457, acc: 0.990, f1: 0.986 | Val loss: 0.0375, acc: 0.992, f1: 0.989
Epoch 009 | Train loss: 0.0423, acc: 0.991, f1: 0.987 | Val loss: 0.0407, acc: 0.992, f1: 0.989
Epoch 010 | Train loss: 0.0399, acc: 0.990, f1: 0.987 | Val loss: 0.0317, acc: 0.992, f1: 0.989
Epoch 011 | Train loss: 0.0386, acc: 0.9

In [9]:
# ========= 9. FINAL TEST EVALUATION =========
test_loss, test_acc, test_f1 = run_epoch(model, test_loader, optimizer=None)
print(f"Test loss: {test_loss:.4f}, acc: {test_acc:.3f}, f1: {test_f1:.3f}")

# Detailed classification report
all_preds = []
all_targets = []
model.eval()
with torch.no_grad():
    for X_num_batch, X_cat_batch, y_batch in test_loader:
        logits = model(X_num_batch.to(DEVICE), X_cat_batch.to(DEVICE))
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

print("Classification report on TEST set:")
print(classification_report(all_targets, all_preds, target_names=label_encoder.classes_))

Test loss: 0.1139, acc: 0.993, f1: 0.990
Classification report on TEST set:
              precision    recall  f1-score   support

       allow       1.00      1.00      1.00      5646
        deny       1.00      0.97      0.98      2257
        drop       0.98      1.00      0.99      1927

    accuracy                           0.99      9830
   macro avg       0.99      0.99      0.99      9830
weighted avg       0.99      0.99      0.99      9830



In [10]:
# ========= 9. PLOTS (LOSS, F1, CONFUSION MATRIX) =========
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

os.makedirs("results", exist_ok=True)

epochs_range = range(1, EPOCHS + 1)

# ---- Loss Plot ----
plt.figure(figsize=(8,5))
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.legend()
plt.savefig("results/train_val_loss.png", dpi=300, bbox_inches="tight")
plt.close()

# ---- F1 Plot ----
plt.figure(figsize=(8,5))
plt.plot(epochs_range, train_f1_list, label="Train F1")
plt.plot(epochs_range, val_f1_list, label="Val F1")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("Train vs Validation F1 Score")
plt.legend()
plt.savefig("results/train_val_f1.png", dpi=300, bbox_inches="tight")
plt.close()

# ---- Confusion Matrix ----
# Test verisi ile tahmin
all_preds = []
all_targets = []

model.eval()
with torch.no_grad():
    for Xn, Xc, yb in test_loader:
        logits = model(Xn.to(DEVICE), Xc.to(DEVICE))
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.savefig("results/confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.close()

print("\nAll plots saved in 'results/' folder.")




All plots saved in 'results/' folder.
